In [24]:
import h5py
import numpy as np



PL = -1
file_path = f"/Users/hariprashadravikumar/sivers_TMD_PhD_project/save_h5_A12B_A2B/ReA2B_PL{PL}_jackknife_data.h5"


with h5py.File(file_path, "r") as f:
    raw_data = f["Dataset1"][:]

kinematics = raw_data[:, 0:3] 

jk_samples = raw_data[:, 3:]   



Kinematics shape: (726, 3)
Samples shape: (726, 2903)
Covariance matrix shape: (726, 726)


In [1]:
%pip install gvar lsqfit
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [46]:
import h5py
import numpy as np
import gvar as gv
import lsqfit

# ---------------------------------------------------------
# 1. Jackknife Statistics Helper
# ---------------------------------------------------------
def Jackknife(datalist): 
    N = len(datalist)
    theta_bar = np.mean(datalist)
    theta_nminus_theta_bar = []
    for i in range(N): 
        theta_n = datalist[i]
        theta_nminus_theta_bar.append(np.square(theta_n - theta_bar))
    sigma_sq = ((N-1)/N) * np.sum(theta_nminus_theta_bar)
    return theta_bar, np.sqrt(sigma_sq)

# ---------------------------------------------------------
# 2. Data Loading & Filtering Helper
# ---------------------------------------------------------
def load_and_filter_data(filepath, bminForFit, etamin, etamax):
    """Loads HDF5 data and applies kinematic cuts."""
    with h5py.File(filepath, "r") as f:
        raw_data = f["Dataset1"][:]
        
    kin = raw_data[:, 0:3]
    samples = raw_data[:, 3:]
    
    eta = kin[:, 0]
    bL  = kin[:, 1]
    bT  = kin[:, 2]
    
    # Calculate b = sqrt(bL^2 + bT^2)
    b_mag = np.sqrt(bL**2 + bT**2)
    
    # Create a boolean mask for our cuts
    mask = (eta >= etamin) & (eta <= etamax) & (b_mag >= bminForFit)
    
    # Return only the rows that passed the cuts
    return kin[mask], samples[mask]

# ---------------------------------------------------------
# 3. Main Simultaneous Fitting Function
# ---------------------------------------------------------
def FitSimultaneousA2B(PL, bminForFit, etamin, etamax):
    print(f"--- Starting Fit: PL={PL}, b_min>={bminForFit}, eta in [{etamin}, {etamax}] ---")
    
    # 1. Construct file paths (Update the base path if needed)
    base_path = "/Users/hariprashadravikumar/sivers_TMD_PhD_project/save_h5_A12B_A2B"
    file_re = f"{base_path}/ReA2B_PL{PL}_jackknife_data.h5"
    file_im = f"{base_path}/ImA2B_PL{PL}_jackknife_data.h5"
    
    # 2. Load and Filter Data
    kin_re, jk_re = load_and_filter_data(file_re, bminForFit, etamin, etamax)
    kin_im, jk_im = load_and_filter_data(file_im, bminForFit, etamin, etamax)
    
    N_re = jk_re.shape[0]
    N_im = jk_im.shape[0]
    N_samples = jk_re.shape[1]
    
    print(f"Data passing cuts -> Real points: {N_re}, Imag points: {N_im}")
    
    # Safety check
    if N_re == 0 or N_im == 0:
        print("Error: No data passed the kinematic cuts!")
        return

    # 3. Calculate the Full Jackknife Covariance Matrix
    combined_jk = np.vstack([jk_re, jk_im])
    mean_data = np.mean(combined_jk, axis=1)
    diff = combined_jk - mean_data[:, None]
    cov_matrix = ((N_samples - 1) / N_samples) * (diff @ diff.T)

    def simultaneous_model(x, p):
        ans = {}
        
        # 1. Extract Shared Parameters (Common to both Re and Im)
        k1, k2 = p['k1'], p['k2']
        f = p['f']
        
        if 'Re' in x:
            x1, x2, x3 = x['Re'][:, 0], x['Re'][:, 1], x['Re'][:, 2]
            
            # 2. Extract Independent Parameters for the Real Part
            a_re = p['a_re']
            c_re, d_re = p['c_re'], p['d_re']
            j_re = p['j_re']
            
            denom = (1 + (d_re + c_re * (1 - k1/x1 + k2/np.sqrt(x1))) * x2**2 + d_re * x3**2)**j_re
            ans['Re'] = a_re * np.exp(-f * x1) / denom
    
        if 'Im' in x:
            x1, x2, x3 = x['Im'][:, 0], x['Im'][:, 1], x['Im'][:, 2]
            
            # 3. Extract Independent Parameters for the Imaginary Part
            a_im = p['a_im']
            c_im, d_im = p['c_im'], p['d_im']
            j_im = p['j_im']
            
            denom = (1 + (d_im + c_im * (1 - k1/x1 + k2/np.sqrt(x1))) * x2**2 + d_im * x3**2)**j_im
            ans['Im'] = a_im * np.exp(-f * x1) * x2 / denom
            
        return ans

    # ---------------------------------------------------------
    # Set up Priors
    # ---------------------------------------------------------
    priors = gv.BufferDict()

    # Shared parameters
    priors['k1']   = gv.gvar(-5.83, 10.0)
    priors['k2']   = gv.gvar(-4.86, 10.0)
    priors['f']    = gv.gvar(0.50,  10.0)

    # Independent parameters (Real Part)
    priors['a_re'] = gv.gvar(4.01,  10.0)
    priors['c_re'] = gv.gvar(0.60,  10.0)
    priors['d_re'] = gv.gvar(0.094, 10.0)
    priors['j_re'] = gv.gvar(2.06,  10.0)

    # Independent parameters (Imaginary Part)
    priors['a_im'] = gv.gvar(0.100, 10.0)
    priors['c_im'] = gv.gvar(0.60,  10.0)
    priors['d_im'] = gv.gvar(0.094, 10.0)
    priors['j_im'] = gv.gvar(2.06,  10.0)
    # 4. Define the Simultaneous Model


    # 6. Perform the Fit at Each Jackknife Sample
    x_dict = {'Re': kin_re, 'Im': kin_im}
    fitted_params = {key: [] for key in priors.keys()}
    
    # We will also track chi-square per degree of freedom
    chi2_dof_list = []

    for i in range(N_samples):
        y_sample_i = combined_jk[:, i]
        y_gvar_i = gv.gvar(y_sample_i, cov_matrix)
        
        y_dict = {
            'Re': y_gvar_i[:N_re],
            'Im': y_gvar_i[N_re:]
        }
        
        fit = lsqfit.nonlinear_fit(data=(x_dict, y_dict), prior=priors, fcn=simultaneous_model)
        
        for key in fitted_params:
            fitted_params[key].append(fit.pmean[key])
            
        chi2_dof_list.append(fit.chi2 / fit.dof)
        if (i + 1) % 500 == 0:
            print(f"  Finished {i + 1} / {N_samples} fits...")

    # 7. Apply Jackknife and Print Results
    print("\n--- Final Extracted Parameters ---")
    for key in fitted_params:
        mean_val, err_val = Jackknife(fitted_params[key])
        print(f"{key:>5} = {mean_val:8.4f} +/- {err_val:8.4f}")
        
    print(f"\nAverage chi^2/dof across samples: {np.mean(chi2_dof_list):.3f}\n")
    
    # Return the dictionary of jackknifed parameter arrays in case you want to plot them later
    return fitted_params

In [48]:
FitSimultaneousA2B(PL=-1, bminForFit=3, etamin=6, etamax=10)

--- Starting Fit: PL=-1, b_min>=3, eta in [6, 10] ---
Data passing cuts -> Real points: 225, Imag points: 230
  Finished 500 / 2903 fits...
  Finished 1000 / 2903 fits...
  Finished 1500 / 2903 fits...
  Finished 2000 / 2903 fits...
  Finished 2500 / 2903 fits...

--- Final Extracted Parameters ---
   k1 =  -5.3997 +/-   0.1726
   k2 =  -4.6586 +/-   0.0716
    f =   0.4343 +/-   0.0050
 a_re =   2.3934 +/-   0.1242
 c_re =   0.7994 +/-   0.1552
 d_re =   0.1046 +/-   0.0096
 j_re =   1.7800 +/-   0.0603
 a_im =   0.0000 +/-   0.0000
 c_im =   0.6000 +/-   0.0000
 d_im =   0.0940 +/-   0.0000
 j_im =   2.0600 +/-   0.0000

Average chi^2/dof across samples: 2.016



{'k1': [-5.399674263893748,
  -5.399998358251994,
  -5.399657856170749,
  -5.403173374775023,
  -5.397019163297741,
  -5.40123730731508,
  -5.399252350056304,
  -5.399401337780528,
  -5.403049168209779,
  -5.4006088359339675,
  -5.408056534067401,
  -5.402874806877741,
  -5.400613369062093,
  -5.402013295163623,
  -5.396386263603598,
  -5.401696725116194,
  -5.395285827520067,
  -5.399680066736858,
  -5.397769987091294,
  -5.402680823868288,
  -5.401975111207592,
  -5.39464055992294,
  -5.394604146822866,
  -5.396056676741491,
  -5.395232515925546,
  -5.405217059586051,
  -5.399258062824815,
  -5.396155603350583,
  -5.400698875673711,
  -5.400228034371319,
  -5.402072142677163,
  -5.398283039538905,
  -5.4023833862514286,
  -5.396442866337711,
  -5.39840086054598,
  -5.399807094159228,
  -5.396856322740843,
  -5.400721056286691,
  -5.399070155562671,
  -5.397994867314449,
  -5.39912553658411,
  -5.400184830588518,
  -5.398266560235565,
  -5.400143883081006,
  -5.399126847863157,
  -5.4

In [3]:
import h5py
import numpy as np
import gvar as gv
import lsqfit
from tqdm.auto import tqdm

# ---------------------------------------------------------
# 1. Jackknife Statistics Helper
# ---------------------------------------------------------
def Jackknife(datalist): 
    N = len(datalist)
    theta_bar = np.mean(datalist)
    theta_nminus_theta_bar = []
    for i in range(N): 
        theta_n = datalist[i]
        theta_nminus_theta_bar.append(np.square(theta_n - theta_bar))
    sigma_sq = ((N-1)/N) * np.sum(theta_nminus_theta_bar)
    return theta_bar, np.sqrt(sigma_sq)

def fmt_err(mean, err):
    # switch to scientific if very small/big
    if mean and (abs(mean) < 1e-3 or abs(mean) >= 1e3):
        m_str = f"{mean:.2e}"           # e.g. "2.86e-08"
        mant, exp = m_str.split("e")
        ndec = len(mant.split(".")[1])  # digits in mantissa
        err_int = int(round(err / 10**int(exp) * 10**ndec))
        return f"{mant}({err_int})e{int(exp)}"
    else:
        ndec = 4                        # choose 4 decimal places
        m_str = f"{mean:.{ndec}f}"      # e.g. "0.4721"
        err_int = int(round(err * 10**ndec))
        return f"{m_str}({err_int:0{ndec}d})"

# ---------------------------------------------------------
# 2. Data Loading & Filtering Helper
# ---------------------------------------------------------
def load_and_filter_data(filepath, bminForFit, etamin, etamax):
    """Loads HDF5 data and applies kinematic cuts."""
    with h5py.File(filepath, "r") as f:
        raw_data = f["Dataset1"][:]
        
    kin = raw_data[:, 0:3]
    samples = raw_data[:, 3:]
    
    eta = kin[:, 0]
    bL  = kin[:, 1]
    bT  = kin[:, 2]
    
    # Calculate b = sqrt(bL^2 + bT^2)
    b_mag = np.sqrt(bL**2 + bT**2)
    
    # Create a boolean mask for our cuts
    mask = (eta >= etamin) & (eta <= etamax) & (b_mag >= bminForFit)
    
    # Return only the rows that passed the cuts
    return kin[mask], samples[mask]


def ReA2B(eta, bL, bT, a, c, d, f, k1, k2, j):
    # Note: Used ** for exponentiation, not ^
    denom = (1 + (d + c * (1 - k1/eta + k2/np.sqrt(eta))) * bL**2 + d * bT**2)**j
    return a * np.exp(-f * eta) / denom

def ImA2B(eta, bL, bT, a, c, d, f, k1, k2, j):
    # Note: Includes the extra bL (*x2) in the numerator
    denom = (1 + (d + c * (1 - k1/eta + k2/np.sqrt(eta))) * bL**2 + d * bT**2)**j
    return a * bL * np.exp(-f * eta) / denom

def fcn(x, p):
    # 1. Unpack the kinematic arrays from the dictionary x
    kin_re = x['Re']
    kin_im = x['Im']
    
    eta_re, bL_re, bT_re = kin_re[:, 0], kin_re[:, 1], kin_re[:, 2]
    eta_im, bL_im, bT_im = kin_im[:, 0], kin_im[:, 1], kin_im[:, 2]

    # 2. Unpack the fit parameters from the dictionary p
    k1, k2, f = p['k1'], p['k2'], p['f']
    
    a_re, c_re, d_re, j_re = p['a_re'], p['c_re'], p['d_re'], p['j_re']
    a_im, c_im, d_im, j_im = p['a_im'], p['c_im'], p['d_im'], p['j_im']

    # 3. Compute the model predictions
    val_re = ReA2B(eta_re, bL_re, bT_re, a_re, c_re, d_re, f, k1, k2, j_re)
    val_im = ImA2B(eta_im, bL_im, bT_im, a_im, c_im, d_im, f, k1, k2, j_im)

    # 4. Return the dictionary of predictions
    return {'Re': val_re, 'Im': val_im}
    
# ---------------------------------------------------------
# 3. Main Simultaneous Fitting Function
# ---------------------------------------------------------
def FitSimultaneousA2B(PL, bminForFit, etamin, etamax):
    print(f"--- Starting Fit: PL={PL}, b_min>={bminForFit}, eta in [{etamin}, {etamax}] ---")
    
    # 1. Construct file paths (Update the base path if needed)
    base_path = "/Users/hariprashadravikumar/sivers_TMD_PhD_project/save_h5_A12B_A2B"
    file_re = f"{base_path}/ReA2B_PL{PL}_jackknife_data.h5"
    file_im = f"{base_path}/ImA2B_PL{PL}_jackknife_data.h5"
    
    # 2. Load and Filter Data
    kin_re, jk_re = load_and_filter_data(file_re, bminForFit, etamin, etamax)
    kin_im, jk_im = load_and_filter_data(file_im, bminForFit, etamin, etamax)
    
    N_re = jk_re.shape[0]
    N_im = jk_im.shape[0]
    N_samples = jk_re.shape[1]
    
    print(f"Data passing cuts -> Real points: {N_re}, Imag points: {N_im}")

    # 3. Calculate the Full Jackknife Covariance Matrix
    combined_jk = np.vstack([jk_re, jk_im])
    mean_data = np.mean(combined_jk, axis=1)
    diff = combined_jk - mean_data[:, None]
    cov_matrix = ((N_samples - 1) / N_samples) * (diff @ diff.T)


    # ---------------------------------------------------------
    # Set up Priors
    # ---------------------------------------------------------
    current_prior = {
     'k1'  : gv.gvar(-5.83, 10.0),
     'k2'  : gv.gvar(-4.86, 10.0),
     'f'   : gv.gvar(0.50,  10.0),
     'a_re' : gv.gvar(4.01,  10.0),
     'c_re' : gv.gvar(0.60,  10.0),
     'd_re' : gv.gvar(0.094, 10.0),
     'j_re' : gv.gvar(2.06,  10.0),
     'a_im' : gv.gvar(0.100, 10.0),
     'c_im' : gv.gvar(0.60,  10.0),
     'd_im' : gv.gvar(0.094, 10.0),
     'j_im' : gv.gvar(2.06,  10.0),
    }


    x_dict = {'Re': kin_re, 'Im': kin_im}
    param_names = ['k1','k2','f','a_re','c_re','d_re','j_re','a_im','c_im','d_im','j_im']
    fitted_params = {key: [] for key in param_names}
    
    # We will also track chi-square per degree of freedom
    chi2_dof_list = []

    for i in tqdm(range(N_samples)):
        y_sample_i = combined_jk[:, i]
        y_gvar_i = gv.gvar(y_sample_i, cov_matrix)
        
        y_dict = {
            'Re': y_gvar_i[:N_re],
            'Im': y_gvar_i[N_re:]
        }
        
        fit = lsqfit.nonlinear_fit(data=(x_dict, y_dict), prior=current_prior, fcn=fcn, debug=False, svdcut=1e-12)
        
        for key in fitted_params:
            fitted_params[key].append(fit.pmean[key])
            
        chi2_dof_list.append(fit.chi2 / fit.dof)

    # 7. Apply Jackknife and Print Results
    print("\n--- Final Extracted Parameters ---")
    for key in fitted_params:
        mean_val, err_val = Jackknife(fitted_params[key])
        print(f"{key:>5} = {fmt_err(mean_val, err_val)}")
        
    mean_chi2, err_chi2 = Jackknife(chi2_dof_list)
    print(f"\nAverage chi^2/dof across samples: {fmt_err(mean_chi2, err_chi2)}\n")
    
    # Return the dictionary of jackknifed parameter arrays in case you want to plot them later
    return fitted_params

In [ ]:
results_1 = FitSimultaneousA2B(PL=-1, bminForFit=3, etamin=6, etamax=10)

--- Starting Fit: PL=-1, b_min>=3, eta in [6, 10] ---
Data passing cuts -> Real points: 225, Imag points: 230


  0%|          | 0/2903 [00:00<?, ?it/s]